In [72]:
import pandas as pd
import numpy as np
import pickle

from tensorflow.keras.models import load_model

In [73]:
model = load_model(r"C:\Customer Churn Prediction ANN\model\model.h5")

In [74]:
with open(r"C:\Customer Churn Prediction ANN\artifacts\scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

with open(r"C:\Customer Churn Prediction ANN\artifacts\label_encoder_gender.pkl", "rb") as f:
    label_encoder_gender = pickle.load(f)

with open(r"C:\Customer Churn Prediction ANN\artifacts\onehot_encoder_geo.pkl", "rb") as f:
    encoder = pickle.load(f)

C:\Users\dell\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:440: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\dell\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\dell\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:440: InconsistentVersionWarning: Trying to unpickle estimator On

In [75]:
input_data = {
    "CreditScore": 600,
    "Geography": "France",
    "Gender": "Male",
    "Age": 40,
    "Tenure": 3,
    "Balance": 60000,
    "NumOfProducts": 2,
    "HasCrCard": 1,
    "IsActiveMember": 1,
    "EstimatedSalary": 50000
}

df = pd.DataFrame([input_data])
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [76]:
df['Gender'] = label_encoder_gender.transform(df['Gender'])

In [77]:
geo_encoded = encoder.transform(df[['Geography']])

geo_df = pd.DataFrame(
    geo_encoded,
    columns=encoder.get_feature_names_out(['Geography'])
)

df = df.drop('Geography', axis=1)
df = pd.concat([df, geo_df], axis=1)

In [78]:
columns_order = [
    'CreditScore', 'Gender', 'Age', 'Tenure', 'Balance',
    'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary',
    'Geography_France', 'Geography_Germany', 'Geography_Spain'
]

df = df.reindex(columns=columns_order, fill_value=0)

In [79]:
num_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
            'NumOfProducts', 'EstimatedSalary']

df[num_cols] = scaler.transform(df[num_cols])

In [80]:
prob = model.predict(df)
pred = (prob > 0.5).astype(int)

print("Churn Probability:", prob[0][0])
print("Prediction:", "Churn" if pred[0][0] == 1 else "No Churn")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
Churn Probability: 0.054999884
Prediction: No Churn


In [81]:
test_data = pd.DataFrame([
    {
        "CreditScore": 400,
        "Geography": "Germany",
        "Gender": "Female",
        "Age": 50,
        "Tenure": 1,
        "Balance": 120000,
        "NumOfProducts": 1,
        "HasCrCard": 0,
        "IsActiveMember": 0,
        "EstimatedSalary": 30000
    },
    {
        "CreditScore": 750,
        "Geography": "Spain",
        "Gender": "Male",
        "Age": 30,
        "Tenure": 5,
        "Balance": 0,
        "NumOfProducts": 2,
        "HasCrCard": 1,
        "IsActiveMember": 1,
        "EstimatedSalary": 90000
    }
])

test_data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,400,Germany,Female,50,1,120000,1,0,0,30000
1,750,Spain,Male,30,5,0,2,1,1,90000


In [82]:
test_data['Gender'] = label_encoder_gender.transform(test_data['Gender'])

In [83]:
geo_encoded = encoder.transform(test_data[['Geography']])

geo_df = pd.DataFrame(
    geo_encoded,
    columns=encoder.get_feature_names_out(['Geography'])
)

test_data = test_data.drop('Geography', axis=1)
test_data = pd.concat([test_data, geo_df], axis=1)

In [84]:
columns_order = [
    'CreditScore', 'Gender', 'Age', 'Tenure', 'Balance',
    'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary',
    'Geography_France', 'Geography_Germany', 'Geography_Spain'
]

test_data = test_data.reindex(columns=columns_order, fill_value=0)

In [85]:
num_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
            'NumOfProducts', 'EstimatedSalary']

test_data[num_cols] = scaler.transform(test_data[num_cols])

In [86]:
probs = model.predict(test_data)
preds = (probs > 0.5).astype(int)

for i in range(len(test_data)):
    print(f"Customer {i+1}")
    print("Probability:", probs[i][0])
    print("Prediction:", "Churn" if preds[i][0] == 1 else "No Churn")
    print("-"*30)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
Customer 1
Probability: 0.9388493
Prediction: Churn
------------------------------
Customer 2
Probability: 0.0040670442
Prediction: No Churn
------------------------------
